# Chapter 31: Overfitting & Regularization

        **Part V - The Training Loop, Mastered** · Companion notebook for *PyTorch From Ground Up, Volume I: Foundations*

        This notebook follows the chapter's progression with fresh, CPU-friendly examples. Type, change, break, and repair the code; the goal is fluency, not passive reading.

        ## Learning objectives

        - Recognize a widening train-validation gap
- Apply dropout and weight decay
- Implement early stopping logic


In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Default device: cpu")


## Create an overfitting opportunity

A high-capacity network trained on few noisy samples can memorize training data while validation stops improving.


In [ ]:
import torch.nn as nn

torch.manual_seed(2)
train_x = torch.randn(40, 10)
train_y = (train_x[:, 0] + 0.4 * torch.randn(40) > 0).long()
val_x = torch.randn(200, 10)
val_y = (val_x[:, 0] + 0.4 * torch.randn(200) > 0).long()
model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.03, weight_decay=1e-3)


## Track both sides

Training and validation must be measured under their correct modes. A gap matters more than either number alone.


In [ ]:
history = {"train": [], "val": []}
best_state = None
best_val = float("inf")
wait, patience = 0, 8
for epoch in range(80):
    model.train()
    train_loss = nn.functional.cross_entropy(model(train_x), train_y)
    optimizer.zero_grad(); train_loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = nn.functional.cross_entropy(model(val_x), val_y)
    history["train"].append(train_loss.item())
    history["val"].append(val_loss.item())
    if val_loss.item() < best_val:
        best_val, wait = val_loss.item(), 0
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
        if wait >= patience:
            break
model.load_state_dict(best_state)
print("epochs:", len(history["train"]), "best val:", best_val)


## Measure generalization

Regularization is successful only if unseen-data performance improves, not merely if training becomes harder.


In [ ]:
model.eval()
with torch.no_grad():
    train_acc = (model(train_x).argmax(1) == train_y).float().mean()
    val_acc = (model(val_x).argmax(1) == val_y).float().mean()
print("train accuracy:", train_acc.item(), "validation accuracy:", val_acc.item())
print("final gap:", history["val"][-1] - history["train"][-1])


## Practice

        Work through these without looking back first.

        1. Train once without weight decay and compare.
2. Try dropout probabilities 0, 0.3, and 0.7.
3. Explain why the best checkpoint is restored after early stopping.

        <details><summary>Study habit</summary>

        Predict every output shape before running a cell. When something fails, read the final line of the error and print the shape, dtype, and device of every tensor involved.

        </details>


In [ ]:
# Your practice space. Replace `pass` with your solution.
pass


## Recap

You completed Chapter 31's companion lab. Before moving on, explain the central idea aloud and recreate the smallest useful example from memory.

**Next:** return to the book for the full explanation, visual reasoning, watch-outs, and chapter exercises.
